# Low-Resource Translation Notebook

## 1. Project Overview

**Task:** Evaluate and compare translation quality for underrepresented languages using LLM-based translation and automated metrics. Explore how quality degrades as languages become less resourced.

**Stack:** `LangChain` + `ChatOllama` + `qwen3.5:9b` for translation, `sacrebleu` for metrics.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Translate** into low-resource languages using LLM |
| 2 | **Compare** high-resource vs low-resource translation quality |
| 3 | **Evaluate** with BLEU, chrF, and LLM-as-judge |
| 4 | **Understand** challenges of low-resource translation |

## 3. Why This Matters

- **7,000+ languages** exist; most have minimal digital resources
- **Translation equality** — low-resource speakers deserve tools too
- **Quality degrades** predictably as training data decreases

## 4. Setup

In [ ]:
import os, json, re, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0)

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

try:
    import sacrebleu
    HAS_SACREBLEU = True
except ImportError:
    HAS_SACREBLEU = False

print(f"LLM ready: {ask('Reply with one word.', 'Say ready.')[:20]}")
print(f"sacrebleu: {HAS_SACREBLEU}")

## 5. Test Sentences

In [ ]:
TEST_SENTENCES = [
    "Hello, how are you today?",
    "Machine learning is transforming healthcare.",
    "The weather is beautiful this morning.",
    "Please send me the report by Friday.",
    "Education is the key to a better future.",
]

LANGUAGE_TIERS = {
    "High-resource": [("Spanish", "es"), ("French", "fr"), ("German", "de")],
    "Medium-resource": [("Turkish", "tr"), ("Vietnamese", "vi"), ("Thai", "th")],
    "Low-resource": [("Swahili", "sw"), ("Yoruba", "yo"), ("Amharic", "am")],
}

print("Test sentences:")
for s in TEST_SENTENCES:
    print(f"  {s}")
print(f"\nLanguage tiers: {sum(len(v) for v in LANGUAGE_TIERS.values())} languages across 3 tiers")

---
## 6. LLM Translation Across Resource Tiers

In [ ]:
TRANSLATE_SYSTEM = "Translate to {language}. Return ONLY the translation. /no_think"

print("LLM TRANSLATIONS BY RESOURCE TIER")
print("=" * 70)
translations = {}
for tier, langs in LANGUAGE_TIERS.items():
    print(f"\n--- {tier} ---")
    for lang_name, lang_code in langs:
        translations[lang_code] = []
        for sent in TEST_SENTENCES[:3]:
            trans = ask(TRANSLATE_SYSTEM.format(language=lang_name), sent)
            translations[lang_code].append(trans)
        print(f"  {lang_name}: {translations[lang_code][0][:60]}...")

## 7. Quality Evaluation (LLM-as-Judge)

In [ ]:
EVAL_SYSTEM = """Score this translation 1-5:
- accuracy: meaning preserved? (1=wrong, 5=perfect)
- fluency: natural in target language? (1=broken, 5=native-like)
Return JSON: {"accuracy":N,"fluency":N,"overall":N,"note":"comment"} /no_think"""

print("TRANSLATION QUALITY BY TIER")
print("=" * 60)
tier_scores = {}
for tier, langs in LANGUAGE_TIERS.items():
    tier_scores[tier] = []
    for lang_name, lang_code in langs:
        if lang_code in translations and translations[lang_code]:
            resp = ask(EVAL_SYSTEM, f"Source (English): {TEST_SENTENCES[0]}\nTranslation ({lang_name}): {translations[lang_code][0]}")
            text = resp
            if "```" in text: text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
            s, e = text.find("{"), text.rfind("}") + 1
            if s >= 0 and e > s:
                try:
                    sc = json.loads(text[s:e])
                    overall = sc.get("overall", (sc.get("accuracy",0)+sc.get("fluency",0))/2)
                    tier_scores[tier].append(overall)
                    print(f"  {lang_name:<12} [{tier:<15}] accuracy={sc.get('accuracy','?')}/5 fluency={sc.get('fluency','?')}/5")
                except: pass

print("\nTIER AVERAGES:")
for tier, scores in tier_scores.items():
    if scores:
        print(f"  {tier:<18}: {sum(scores)/len(scores):.1f}/5")

## 8. Automated Metrics (BLEU/chrF)

In [ ]:
if HAS_SACREBLEU:
    BACK_SYSTEM = "Translate this to English. Return ONLY the translation. /no_think"
    print("ROUND-TRIP TRANSLATION (proxy for quality)")
    print("=" * 60)
    for tier, langs in LANGUAGE_TIERS.items():
        for lang_name, lang_code in langs[:1]:
            if lang_code in translations and translations[lang_code]:
                back_trans = ask(BACK_SYSTEM, translations[lang_code][0])
                bleu = sacrebleu.sentence_bleu(back_trans, [TEST_SENTENCES[0]]).score
                chrf = sacrebleu.sentence_chrf(back_trans, [TEST_SENTENCES[0]]).score
                print(f"  {lang_name:<12} [{tier:<15}] BLEU={bleu:.1f} chrF={chrf:.1f}")
                print(f"    Original:   {TEST_SENTENCES[0]}")
                print(f"    Round-trip: {back_trans[:60]}")
else:
    print("sacrebleu not installed -- skipping automated metrics")

## 9. Challenges Analysis

In [ ]:
CHALLENGE_SYSTEM = """Analyze translation challenges for {language}.
Return JSON: {"resource_level": "high|medium|low", "challenges": ["challenge1"], "data_availability": "description", "quality_expectation": "description"} /no_think"""

print("LOW-RESOURCE CHALLENGES")
print("=" * 60)
for lang_name, lang_code in LANGUAGE_TIERS["Low-resource"]:
    resp = ask(CHALLENGE_SYSTEM.format(language=lang_name), f"Analyze translation challenges for {lang_name}")
    text = resp
    if "```" in text: text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
    s, e = text.find("{"), text.rfind("}") + 1
    if s >= 0 and e > s:
        try:
            result = json.loads(text[s:e])
            challenges = result.get("challenges", [])[:3]
            print(f"\n  {lang_name}:")
            for c in challenges:
                print(f"    - {c}")
        except: pass

## 10. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| Assuming equal quality across languages | Always evaluate per-language |
| BLEU only for low-resource | Use chrF + human eval for morphologically rich languages |
| Ignoring script differences | Some languages need special tokenization |
| No baseline comparison | Compare against round-trip translation |

| # | Takeaway |
|---|----------|
| 1 | **Translation quality drops predictably** with resource level |
| 2 | **LLMs handle high-resource languages** well, struggle with low-resource |
| 3 | **Round-trip translation** is a useful proxy for quality |
| 4 | **chrF is better than BLEU** for morphologically rich languages |
| 5 | **Human evaluation is essential** for low-resource languages |

---
*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*

